In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam

# ==========================================
# 1. DATA SIMULATION (The "Ground Truth")
# ==========================================
def generate_yield_curves(n_samples=1000, n_points=30):
    """
    Generates synthetic 'smooth' yield curves using a simplified Nelson-Siegel style logic.
    These represent the 'true' fundamental value of the asset over time/maturity.
    """
    t = np.linspace(0.1, 30, n_points)
    curves = []

    for _ in range(n_samples):
        # Randomize level, slope, and curvature parameters
        beta0 = np.random.uniform(0.02, 0.05)  # Long term rate
        beta1 = np.random.uniform(-0.02, 0.02) # Short term spread
        beta2 = np.random.uniform(-0.02, 0.02) # Hump
        tau = np.random.uniform(1.0, 5.0)      # Decay factor

        # Simplified curve function
        curve = beta0 + beta1 * ((1 - np.exp(-t/tau)) / (t/tau)) + \
                beta2 * (((1 - np.exp(-t/tau)) / (t/tau)) - np.exp(-t/tau))
        curves.append(curve)

    return np.array(curves)

# ==========================================
# 2. DATA CORRUPTION (The "Dirty Data")
# ==========================================
def add_financial_noise(curves, outlier_prob=0.05, noise_level=0.002):
    """
    Corrupts the smooth curves with realistic financial data issues:
    1. Gaussian Noise (Market Microstructure noise)
    2. Spikes (Fat-finger errors / Illiquid quotes)
    3. Dropouts (Missing feeds, represented as 0)
    """
    noisy_curves = curves.copy()
    rows, cols = curves.shape

    # 1. Add Gaussian Noise (White noise)
    noise = np.random.normal(0, noise_level, (rows, cols))
    noisy_curves += noise

    # 2. Add Outliers (Spikes)
    # Randomly select indices to corrupt
    mask = np.random.rand(rows, cols) < outlier_prob
    # Add large random shocks (positive or negative)
    random_spikes = np.random.choice([-0.02, 0.02], size=(rows, cols)) * np.random.uniform(1, 3, (rows, cols))
    noisy_curves[mask] += random_spikes[mask]

    return noisy_curves

# ==========================================
# 3. MAIN EXECUTION FLOW
# ==========================================

# Settings
N_SAMPLES = 5000
N_POINTS = 30 # E.g., maturities from 1Y to 30Y

print(f"Generating {N_SAMPLES} synthetic yield curves...")
clean_data = generate_yield_curves(N_SAMPLES, N_POINTS)

print("Injecting noise, outliers, and artifacts...")
noisy_data = add_financial_noise(clean_data)

# Split into Train/Test
split_idx = int(0.8 * N_SAMPLES)
x_train_noisy = noisy_data[:split_idx]
x_train_clean = clean_data[:split_idx]
x_test_noisy = noisy_data[split_idx:]
x_test_clean = clean_data[split_idx:]

# ==========================================
# 4. BUILD THE AUTOENCODER MODEL
# ==========================================
# Concept: Compressing input into a latent space forces the model to learn
# the "manifold" of plausible yield curves, ignoring random noise.

model = Sequential([
    Input(shape=(N_POINTS,)),

    # Encoder: Compress dimensions
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),

    # Bottleneck (Latent Space)
    # The model must represent the entire curve using only 8 numbers.
    # Noise is random and hard to compress; structure is easy to compress.
    Dense(8, activation='relu', name='bottleneck'),

    # Decoder: Reconstruct dimensions
    Dense(32, activation='relu'),
    Dense(64, activation='relu'),

    # Output Layer (Linear activation for regression)
    Dense(N_POINTS, activation='linear')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nTraining Denoising Autoencoder...")
# Note: Input is NOISY, Target is CLEAN. This is the key to DAEs.
history = model.fit(
    x_train_noisy, x_train_clean,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.1,
    verbose=0 # Set to 1 to see progress logs
)
print("Training Complete.")

# ==========================================
# 5. VISUALIZATION
# ==========================================
print("Predicting denoised curves...")
denoised_data = model.predict(x_test_noisy)

def plot_results(index):
    plt.figure(figsize=(10, 6))

    # Plot Ground Truth
    plt.plot(x_test_clean[index], 'g--', linewidth=2, label='True Manifold (Clean)')

    # Plot Noisy Input
    plt.plot(x_test_noisy[index], 'r-', alpha=0.5, linewidth=1, label='Input (Dirty/Noisy)')
    plt.scatter(range(N_POINTS), x_test_noisy[index], c='red', s=10, alpha=0.5)

    # Plot Denoised Output
    plt.plot(denoised_data[index], 'b-', linewidth=3, label='Reconstruction (Denoised)')

    plt.title(f'Denoising Autoencoder Result (Sample #{index})')
    plt.xlabel('Maturity Point')
    plt.ylabel('Yield')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Show 3 random examples from the test set
print("Displaying results...")
indices = np.random.choice(len(x_test_noisy), 3, replace=False)
for idx in indices:
    plot_results(idx)

# Plot Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Training History')
plt.ylabel('MSE Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()